In [ ]:
# no penalty
from utils_monoBP_single import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import math
import pandas as pd
import time
import sys
import json
from torch.utils.data import Dataset
from pathlib import Path

# prior for theta0
a0 = -float('inf') # -5.0
b0 = float('inf')  # 5.0
# prior for theta1-thetaM
a = -float('inf') # 0.0
b = float('inf') # 1.0

M = 10

x_dim = 2
obs_size = 100

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(torch.version.cuda) 
print(torch.cuda.is_available()) 

In [ ]:
def sample_trunc_fast(mean_theta, prop_cov, lower, upper, sample_size, batch_size=10000):
    dist = torch.distributions.MultivariateNormal(
        loc=mean_theta,
        covariance_matrix=prop_cov
    )

    dim = lower.numel()
    out = torch.empty(sample_size, dim, device=lower.device, dtype=lower.dtype)

    filled = 0

    while filled < sample_size:
        theta = dist.sample((batch_size,))
        mask = ((theta >= lower) & (theta <= upper)).all(dim=1)
        valid = theta[mask]

        n = min(valid.shape[0], sample_size - filled)
        if n > 0:
            out[filled:filled+n] = valid[:n]
            filled += n

    return out

def gen_ref_distinct_theta_nondiag(prop_mean, prop_cov, lower, upper, sample_size = 10000):
    """
        generate theta from a (truncated) gaussian proposal distribution, and then use theta to generate x
        mean_theta: the mean of the truncated normal, a 1-dim tensor of the same length as theta
        std_theta: the std of the truncated normal, a 1-dim tensor of the same length as theta
        lower: the lower bound for each dimension of theta, a 1-dim tensor
        upper: the upper bound for each dimension of theta, a 1-dim tensor
    """
    time_start = time.time()
    
    theta_time1 = time.time()
    theta = sample_trunc_fast(prop_mean, prop_cov, lower, upper, sample_size)
    theta_time2 = time.time()
    print(f"time of generating theta: {round(theta_time2 - theta_time1)} seconds")


    x = torch.rand(sample_size)
    A = get_A(M)
    psi = get_psi(x, M) # (sample_size, M + 1)
    y = ( (psi @ torch.linalg.inv(A)) * theta ).sum(dim = 1) + sigma * torch.randn(sample_size)
    
    
    data = torch.zeros(sample_size, 2)
    data[:, 0] = x
    data[:, 1] = y

    time_end = time.time()
    print(f"Total time for generating ABC table = {round((time_end - time_start) / 60, 3)} minutes")
    return theta, data

In [ ]:
task_id = 0

In [ ]:
theta_pre = torch.tensor(pd.read_csv(f"res_SW1_precond/theta_pre_task{task_id}.csv").values, dtype = torch.float32).contiguous()


lower = torch.zeros(M + 1) # .to(device)
lower[0] = a0
lower[1:] = a
upper = torch.zeros(M + 1) # .to(device)
upper[0] = b0
upper[1:] = b

actual_inf_rate = torch.ones(M + 1)
actual_inf_rate[-2:] = 2

inf_rate = actual_inf_rate


prop_mean = theta_pre.mean(dim = 0)
prop_cov = torch.diag(inf_rate) @ torch.cov(theta_pre.T) @ torch.diag(inf_rate)

## Score on a single data

In [ ]:
model = single_ELU_LikeScoreMatchingNN(theta_dim = M+1, x_dim=2, obs_size=1000, hidden_size = 64, num_layers = 3).to(device)

In [ ]:
ckpt = torch.load(f'model_single_init/checkpoint_task{task_id}_trainsize{int(1e6)}.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

bias_lastlayer = ckpt['bias_lastlayer'].to(device)

with torch.no_grad(): 
    model.layers[-1].bias -= bias_lastlayer

# Generate data from the proposal

In [ ]:
sample_size = int(1e4) 
theta_r0, data_r0 = gen_ref_distinct_theta_nondiag(prop_mean, prop_cov, lower, upper, sample_size)

## Compute true and estimated score

In [ ]:
tmp_size = sample_size
x_r0 = data_r0[:tmp_size, ::2]
y_r0 = data_r0[:tmp_size, 1::2]

true_score = torch.zeros(tmp_size, M + 1)
for i in tqdm(range(tmp_size)):
    psi = get_psi(x_r0[i], M)
    A = get_A(M)
    design = psi @ torch.linalg.inv(A)
    design = design
    true_score[i] = 1/sigma**2 * (design.T @ y_r0[i] - design.T @ design @ theta_r0[i])

In [ ]:
est_score = model.cpu()(theta_r0[:tmp_size], data_r0[:tmp_size]).detach()

In [ ]:
score_error = torch.linalg.norm(est_score - true_score, dim = 1)**2

### Score error v.s. direction of the true score

In [ ]:
# Standardize
std = torch.sqrt(torch.diag(prop_cov))
Sigma_std = prop_cov / (std[:, None] * std[None, :])
true_score_std = true_score * std
est_score_std = est_score * std
score_error_std = torch.linalg.norm(est_score_std - true_score_std, dim = 1)**2

v_std = true_score_std
similarity = torch.einsum("bi,ij,bj->b", v_std, Sigma_std, v_std) / torch.sum(v_std**2, dim=1)

In [ ]:
import matplotlib.pyplot as plt

x = similarity.detach().cpu().numpy()
y = score_error_std.detach().cpu().numpy()

plt.figure(figsize=(5, 4), dpi=300)

plt.plot(x, y, 'o', markersize=3, alpha=0.3)

plt.xlabel("Similarity", fontsize=12)
plt.ylabel("Score error", fontsize=12)

plt.xlim(-0.02, 1.02)
# plt.ylim(0, 10)

plt.tick_params(labelsize=11)
plt.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("similarity_score_error.pdf", bbox_inches="tight")
plt.show()